In [0]:
# Databricks notebook source
# =============================================================================
# PRIMEINSURANCE — UC1: DQ INSIGHT ENGINE
# Reads : primeins.silver.dq_issues_log
# Writes: primeins.gold.dq_explanation_report_test  (test mode)
#         primeins.gold.dq_explanation_report        (production)
#
# PURPOSE:
#   Translates technical DQ logs into compliance-ready business explanations
#   using an LLM. Every row from dq_issues_log is sent to the model with a
#   structured prompt and the response is written back to Gold.
#
# LLM MODES (swap by changing TEST_MODE flag in Section 1):
#   TEST_MODE = True  → OpenRouter / arcee-ai/trinity-large-preview:free
#   TEST_MODE = False → Databricks Foundation Model (databricks-gpt-oss-20b)
#
# OUTPUT TABLE COLUMNS (minimum required by spec):
#   issue_id, table_name, rule_name, consequence, detail, suggested_fix,
#   logged_at, ai_explanation, generation_status, model_name, generated_at,
#   _gold_run_id
# =============================================================================

# COMMAND ----------

# ------------------------------------------------------------------
# 0. IMPORTS & SETUP
# ------------------------------------------------------------------
from openai import OpenAI
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime
import json

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# ------------------------------------------------------------------
# TOGGLE THIS FLAG TO SWITCH LLM BACKEND
#   True  = OpenRouter (testing)
#   False = Databricks Foundation Model (production)
# ------------------------------------------------------------------
TEST_MODE = True

print(f"UC1 DQ Insight Engine  |  run_id    = {RUN_ID}")
print(f"                       |  test_mode = {TEST_MODE}")

# COMMAND ----------

# ------------------------------------------------------------------
# 1. LLM CLIENT SETUP
#
# TWO SETUPS — only the active one is used based on TEST_MODE.
# The other is kept commented so swapping is a one-line change.
# ------------------------------------------------------------------

if TEST_MODE:
    # ── TEST: OpenRouter (free tier, no Databricks billing) ──────────
    client = OpenAI(
        api_key  = "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",
        base_url = "https://openrouter.ai/api/v1"
    )
    MODEL_ID = "arcee-ai/trinity-large-preview:free"
    TARGET_TABLE = "primeins.gold.dq_explanation_report_test"

else:
    # ── PRODUCTION: Databricks Foundation Model ───────────────────────
    # Uncomment this block and set TEST_MODE = False to go live
    #
    # DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    # WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")
    # client = OpenAI(
    #     api_key  = DATABRICKS_TOKEN,
    #     base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
    # )
    MODEL_ID = "databricks-gpt-oss-20b"
    TARGET_TABLE = "primeins.gold.dq_explanation_report"

print(f"Model        : {MODEL_ID}")
print(f"Target table : {TARGET_TABLE}")

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 2. CONFIRM SOURCE TABLE EXISTS AND REVIEW SCHEMA
#
# dq_issues_log columns (from sdp_silver.py):
#   logged_at     : String — ISO timestamp when the issue was logged
#   table_name    : String — which Silver table was affected
#   rule_name     : String — technical rule identifier (e.g. S5_terto_fix)
#   consequence   : String — action taken: "fix", "warn", "quarantine"
#   detail        : String — technical description of what was found
#   suggested_fix : String — recommended remediation from the pipeline
#
# NOTE: column_name, severity, affected_records are not yet in the
# source table — tracked as a Silver pipeline enhancement.
# issue_id is derived here as a synthetic row_number PK.
# ------------------------------------------------------------------
SOURCE_TABLE = "primeins.silver.dq_issues_log"

try:
    dq_raw = spark.table(SOURCE_TABLE)
    print(f"\nSource table : {SOURCE_TABLE}")
    print(f"Row count    : {dq_raw.count()}")
    print(f"Columns      : {dq_raw.columns}\n")
    dq_raw.show(5, truncate=False)
except Exception as e:
    raise RuntimeError(
        f"Source table {SOURCE_TABLE} not found. "
        f"Run the Silver pipeline first.\n{e}"
    )

# COMMAND ----------

# ------------------------------------------------------------------
# 3. ADD SYNTHETIC issue_id
#
# The source table has no natural PK. Row number over logged_at +
# table_name + rule_name is deterministic across runs as long as
# the source data doesn't change — safe as a trace key.
# ------------------------------------------------------------------
dq_issues = dq_raw.withColumn(
    "issue_id",
    F.row_number().over(Window.orderBy("logged_at", "table_name", "rule_name"))
)

dq_rows = dq_issues.collect()
print(f"Issues to process: {len(dq_rows)}")

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 4. PROMPT DESIGN
#
# SYSTEM PROMPT — sets persona and enforces five-section structure.
# Five sections map directly to what compliance teams need:
#   FINDING        → what was found in plain English
#   BUSINESS IMPACT→ why it matters to the business
#   ROOT CAUSE     → what caused it
#   ACTION TAKEN   → what the pipeline did to fix it
#   PREVENTION     → what should be done to stop recurrence
#
# KEY DESIGN DECISIONS:
#   - Persona anchoring ("data governance advisor, not an engineer")
#     locks tone and vocabulary to compliance language
#   - Explicit numbered sections prevent the model from merging
#     all five points into a single unstructured paragraph
#   - "No preamble or sign-off" suppresses "Certainly! Here is..."
#   - consequence is pre-mapped to plain English before the prompt
#     so the model never sees "fix"/"warn"/"quarantine" raw codes
#   - Temperature 0.3 — low for consistent, auditable language
# ------------------------------------------------------------------

SYSTEM_PROMPT = """You are a data governance advisor writing for a compliance officer, not an engineer.

Your job is to explain data quality issues in plain business English.

Rules:
- Never use technical jargon (no "schema", "null", "regex", "pipeline", "DataFrame")
- Write in clear, direct sentences a non-technical reader can act on
- Use exactly these five section headers, in this order:

FINDING:
BUSINESS IMPACT:
ROOT CAUSE:
ACTION TAKEN:
PREVENTION:

Do not combine sections. Do not add a preamble or sign-off.
Respond only with the five sections."""

def build_prompt(row) -> str:
    """
    Builds the user prompt from a single dq_issues_log row.
    consequence is pre-mapped to avoid raw codes reaching the model.
    """
    consequence_map = {
        "fix":        "The issue was automatically corrected by the pipeline",
        "warn":       "The issue was flagged for review but records were not removed",
        "quarantine": "Affected records were moved to a quarantine table for manual review",
    }
    consequence_text = consequence_map.get(
        str(row["consequence"]).lower(),
        str(row["consequence"])
    )

    return f"""A data quality issue was detected in our insurance data system.
Explain it clearly for a compliance officer using the five sections.

SYSTEM AFFECTED   : {row['table_name']}
ISSUE TYPE        : {row['rule_name']}
WHAT HAPPENED     : {row['detail']}
WHAT THE SYSTEM DID: {consequence_text}
RECOMMENDED ACTION: {row['suggested_fix']}"""

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 5. RESPONSE PARSER
#
# databricks-gpt-oss-20b returns structured JSON blocks:
#   [{"type": "reasoning", "summary": [...]},
#    {"type": "text",      "text":    "actual answer"}]
#
# We extract only the "text" block. Returning the raw response would
# expose the model's reasoning chain.
#
# OpenRouter / arcee model returns a plain string — the parser handles
# both formats so swapping backends requires no code change here.
# ------------------------------------------------------------------
def parse_response(raw_content) -> str:
    if isinstance(raw_content, str):
        try:
            blocks = json.loads(raw_content)
            text_blocks = [b["text"] for b in blocks if b.get("type") == "text"]
            return "\n".join(text_blocks).strip() if text_blocks else raw_content.strip()
        except (json.JSONDecodeError, KeyError, TypeError):
            return raw_content.strip()
    return str(raw_content).strip()


In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 6. LLM CALL WRAPPER
#
# Each row is wrapped in try/except so one failed call does not abort
# the batch. Failed rows get status="error" and the error message in
# ai_explanation — the output table is always complete.
# ------------------------------------------------------------------
def call_llm(prompt: str) -> tuple[str, str]:
    try:
        response = client.chat.completions.create(
            model       = MODEL_ID,
            messages    = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            max_tokens  = 800,
            temperature = 0.3,
        )
        raw = response.choices[0].message.content
        return parse_response(raw), "success"
    except Exception as e:
        return f"LLM call failed: {str(e)}", "error"


# COMMAND ----------

# ------------------------------------------------------------------
# 7. PROCESS ALL ROWS
# ------------------------------------------------------------------
results = []
total   = len(dq_rows)

print(f"Processing {total} DQ issues...\n")

for i, row in enumerate(dq_rows, 1):
    print(f"  [{i}/{total}]  {row['table_name']} | {row['rule_name']}")

    prompt              = build_prompt(row)
    explanation, status = call_llm(prompt)

    results.append({
        # ── Original issue fields (preserved verbatim for traceability) ──
        "issue_id":          int(row["issue_id"]),
        "table_name":        str(row["table_name"]),
        "rule_name":         str(row["rule_name"]),
        "consequence":       str(row["consequence"]),
        "detail":            str(row["detail"]),
        "suggested_fix":     str(row["suggested_fix"]),
        "logged_at":         str(row["logged_at"]),
        # ── AI-generated fields ──────────────────────────────────────────
        "ai_explanation":    explanation,
        "generation_status": status,
        "model_name":        MODEL_ID,
        "generated_at":      datetime.now().isoformat(),
        # ── Pipeline audit ───────────────────────────────────────────────
        "_gold_run_id":      RUN_ID,
    })

    print(f"         → {status}")

print(f"\nAll {total} issues processed.")



In [0]:
# ------------------------------------------------------------------
# 8. OUTPUT SCHEMA
#
# Explicit schema prevents Spark from inferring wrong types from Python
# dicts — especially important for long LLM text fields.
# ------------------------------------------------------------------
output_schema = T.StructType([
    T.StructField("issue_id",          T.IntegerType(), nullable=False),
    T.StructField("table_name",        T.StringType(),  nullable=True),
    T.StructField("rule_name",         T.StringType(),  nullable=True),
    T.StructField("consequence",       T.StringType(),  nullable=True),
    T.StructField("detail",            T.StringType(),  nullable=True),
    T.StructField("suggested_fix",     T.StringType(),  nullable=True),
    T.StructField("logged_at",         T.StringType(),  nullable=True),
    T.StructField("ai_explanation",    T.StringType(),  nullable=True),
    T.StructField("generation_status", T.StringType(),  nullable=True),
    T.StructField("model_name",        T.StringType(),  nullable=True),
    T.StructField("generated_at",      T.StringType(),  nullable=True),
    T.StructField("_gold_run_id",      T.StringType(),  nullable=True),
])

In [0]:
# ------------------------------------------------------------------
# 9. WRITE TO GOLD
#
# Full overwrite — idempotent, safe to re-run.
# TARGET_TABLE switches between test and production based on TEST_MODE.
# ------------------------------------------------------------------
output_df = spark.createDataFrame(results, schema=output_schema)

(output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

final_count = spark.table(TARGET_TABLE).count()
print(f"\n✅  {TARGET_TABLE} — {final_count} rows written")

In [0]:
# ------------------------------------------------------------------
# 10. DISPLAY SAMPLE OUTPUT
#
# Shows the first 3 explanations to verify the model returned
# structured compliance language before taking a submission screenshot.
# ------------------------------------------------------------------
print("=" * 70)
print(f"SAMPLE OUTPUT — {TARGET_TABLE}")
print("=" * 70)

sample = spark.table(TARGET_TABLE).limit(3).collect()

for row in sample:
    print(f"\n{'─' * 70}")
    print(f"Issue ID   : {row['issue_id']}")
    print(f"Table      : {row['table_name']}")
    print(f"Rule       : {row['rule_name']}")
    print(f"Consequence: {row['consequence']}")
    print(f"Status     : {row['generation_status']}")
    print(f"Model      : {row['model_name']}")
    print(f"Generated  : {row['generated_at']}")
    print(f"\n{'─' * 35} AI EXPLANATION {'─' * 20}\n")
    print(row['ai_explanation'])

print(f"\n{'─' * 70}")
print(f"Run ID     : {RUN_ID}")
print(f"Total rows : {final_count}")




# ------------------------------------------------------------------
# 11. SCHEMA CONFIRMATION (for submission screenshot)
# ------------------------------------------------------------------
print("\nFinal table schema:")
spark.table(TARGET_TABLE).printSchema()

In [0]:
%sql
SELECT *
FROM `primeins`.`gold`.`dq_explanation_report_test`